In [6]:
from pathlib import Path
from urllib.request import urlretrieve

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-05.parquet"

file_path = data_dir / "yellow_tripdata_2026-05.parquet"

if not file_path.exists():
    urlretrieve(url, file_path)

file_path

PosixPath('data/yellow_tripdata_2026-05.parquet')

In [7]:
import pandas as pd
import pyarrow.parquet as pq

file_path = "data/yellow_tripdata_2026-05.parquet"

df = next(
    pq.ParquetFile(file_path).iter_batches(batch_size=100000)
).to_pandas()

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-05-01 00:04:59,2026-05-01 00:32:48,1,7.51,1,N,138,37,2,35.9,6.00,0.5,0.00,0.0,1.0,45.40,0.0,2.0,0.00
1,2,2026-05-01 00:37:05,2026-05-01 00:59:47,1,6.14,1,N,138,237,1,27.5,6.00,0.5,9.56,0.0,1.0,49.81,2.5,2.0,0.75
2,1,2026-05-01 00:34:05,2026-05-01 00:54:02,1,2.40,1,N,249,232,1,19.1,4.25,0.5,3.73,0.0,1.0,28.58,2.5,0.0,0.75
3,1,2026-05-01 00:55:07,2026-05-01 01:02:43,0,1.20,1,N,232,114,1,9.3,4.25,0.5,3.00,0.0,1.0,18.05,2.5,0.0,0.75
4,7,2026-05-01 00:44:13,2026-05-01 00:44:13,2,0.86,1,N,140,237,1,7.2,0.00,0.5,2.44,0.0,1.0,14.64,2.5,0.0,0.00


In [8]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

print(mlflow.get_tracking_uri())

http://127.0.0.1:5000


In [10]:
mlflow.set_experiment("nyc-taxi-duration")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1795699929285, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795699929285, lifecycle_stage='active', name='nyc-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df["duration"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
)

df["duration"] = df["duration"].dt.total_seconds() / 60

df["duration"].describe()

In [ ]:
initial_count = len(df)

df = df[(df["duration"] >= 1) & (df["duration"] <= 60)].copy()

df.shape

In [ ]:
len(df) / initial_count * 100

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
sns.histplot(df["duration"], bins=60)
plt.xlabel("Durée du trajet (minutes)")
plt.ylabel("Nombre de trajets")
plt.title("Distribution des durées des trajets")
plt.show()

In [ ]:
categorical = ["PULocationID", "DOLocationID"]
numerical = ["trip_distance"]

df[categorical] = df[categorical].astype(str)

df[categorical + numerical].head()

In [ ]:
X = df[categorical + numerical]
y = df["duration"]

print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print("X_train :", X_train.shape)
print("X_val :", X_val.shape)
print("y_train :", y_train.shape)
print("y_val :", y_val.shape)

In [ ]:
from sklearn.feature_extraction import DictVectorizer

dv = DictVectorizer()

train_dicts = X_train.to_dict(orient="records")
val_dicts = X_val.to_dict(orient="records")

X_train_encoded = dv.fit_transform(train_dicts)
X_val_encoded = dv.transform(val_dicts)

In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_encoded, y_train)

In [ ]:
y_pred_lr = lr.predict(X_val_encoded)

y_pred_lr[:10]


In [ ]:
from sklearn.metrics import root_mean_squared_error

rmse_lr = root_mean_squared_error(y_val, y_pred_lr)

print("RMSE Régression linéaire :", rmse_lr)

In [ ]:
import numpy as np
from sklearn.linear_model import Lasso

X_train_encoded = X_train_encoded.tocsc()
X_val_encoded = X_val_encoded.tocsc()

X_train_encoded.indices = X_train_encoded.indices.astype(np.int32)
X_train_encoded.indptr = X_train_encoded.indptr.astype(np.int32)

X_val_encoded.indices = X_val_encoded.indices.astype(np.int32)
X_val_encoded.indptr = X_val_encoded.indptr.astype(np.int32)

lasso = Lasso(alpha=0.01)
lasso.fit(X_train_encoded, y_train)

In [ ]:
y_pred_lasso = lasso.predict(X_val_encoded)

y_pred_lasso[:10]

In [ ]:
rmse_lasso = root_mean_squared_error(y_val, y_pred_lasso)

print("RMSE Lasso :", rmse_lasso)

In [ ]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=2,
    tree_method="hist"
)

xgb_model.fit(X_train_encoded, y_train)

In [ ]:
y_pred_xgb = xgb_model.predict(X_val_encoded)

y_pred_xgb[:10]

In [ ]:
rmse_xgb = root_mean_squared_error(y_val, y_pred_xgb)

print("RMSE XGBoost :", rmse_xgb)

In [ ]:
results = pd.DataFrame({
    "Modèle": ["Régression linéaire", "Lasso", "XGBoost"],
    "RMSE": [rmse_lr, rmse_lasso, rmse_xgb]
})

results.sort_values("RMSE")

In [ ]:
plt.figure(figsize=(8, 5))

sns.barplot(data=results, x="Modèle", y="RMSE")

plt.title("Comparaison des performances des modèles")
plt.ylabel("RMSE (minutes)")
plt.xlabel("Modèle")

plt.show()

In [ ]:
best_model = results.loc[results["RMSE"].idxmin()]

print("Meilleur modèle :", best_model["Modèle"])
print("RMSE :", best_model["RMSE"], "minutes")

In [ ]:
print("Conclusion du TP")

print(f"Régression linéaire : {rmse_lr:.2f} minutes")
print(f"Lasso : {rmse_lasso:.2f} minutes")
print(f"XGBoost : {rmse_xgb:.2f} minutes")

print(f"Le modèle le plus performant est {best_model['Modèle']}")
print(f"Son RMSE est de {best_model['RMSE']:.2f} minutes.")

In [ ]:
with mlflow.start_run():
    mlflow.set_tag("task", "taxi-duration-prediction")

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("sample_size", len(y_train) + len(y_val))
    mlflow.log_param("validation_ratio", 0.2)

    mlflow.log_metric("validation_rmse", rmse_xgb)

    mlflow.sklearn.log_model(
        sk_model=xgb_model,
        name="model",
        skops_trusted_types=[
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBRegressor"
        ]
    )

In [12]:
import os
import mlflow

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow.set_tracking_uri("file:./mlruns")

run_id = "88035c0c81f14e78a95bd75a6c21217c"

run = mlflow.get_run(run_id)

print(run.data.params)
print(run.data.metrics)
print(mlflow.search_logged_models(
    experiment_ids=[run.info.experiment_id]
)[["model_id", "name"]])

{'model': 'XGBoost', 'validation_ratio': '0.2', 'sample_size': '3883752'}
{'validation_rmse': 6.5492056576552}
                             model_id   name
0  m-9c664b13f96a4645b1dbf10e3848d521  model
1  m-5090ba5d6a1c4b9a89a920aa6b262c7b  model


In [13]:
models = mlflow.search_logged_models(
    experiment_ids=[run.info.experiment_id]
)

model_id = models[
    (models["source_run_id"] == run_id)
    & (models["status"] == "READY")
].iloc[0]["model_id"]

xgb_model = mlflow.sklearn.load_model(
    f"models:/{model_id}"
)

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("nyc-taxi-duration")

with mlflow.start_run(run_name="XGBoost-SQLite"):
    mlflow.log_params(run.data.params)
    mlflow.log_metrics(run.data.metrics)
    mlflow.set_tag("task", "taxi-duration-prediction")

    mlflow.sklearn.log_model(
        sk_model=xgb_model,
        name="model",
        skops_trusted_types=[
            "xgboost.core.Booster",
            "xgboost.sklearn.XGBRegressor"
        ]
    )

print("Scénario 2 terminé !")

🏃 View run XGBoost-SQLite at: http://127.0.0.1:5000/#/experiments/1/runs/3db6ffae5cf44a998d9c93b25ddc9197
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
Scénario 2 terminé !


In [14]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment = mlflow.set_experiment("nyc-taxi-duration")

models = mlflow.search_logged_models(
    experiment_ids=[experiment.experiment_id]
)

model_id = models[
    models["status"] == "READY"
].iloc[0]["model_id"]

loaded_model = mlflow.sklearn.load_model(
    f"models:/{model_id}"
)

print("Modèle chargé avec succès !")
print(type(loaded_model))

Modèle chargé avec succès !
<class 'xgboost.sklearn.XGBRegressor'>
